# 02 — Clinical Metadata Preprocessing

## Objective

This notebook develops the preprocessing pipeline for the clinical metadata used in the multimodal models.

The primary metadata features identified during the dataset audit are:

- `age`
- `region`
- `itch`
- `grew`
- `hurt`
- `changed`
- `bleed`
- `elevation`

The preprocessing pipeline will:

- inspect the selected variables and their data types,
- distinguish numerical and categorical features,
- define appropriate encoding and scaling,
- fit preprocessing using training data only within each cross-validation fold,
- verify that the transformed features contain no missing or invalid values,
- provide a reusable preprocessing procedure for all multimodal models.

The fixed patient-grouped folds created in `01_dataset_audit.ipynb` are retained unchanged.

## 1. Setup

Import the libraries required for metadata inspection and preprocessing.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

## 2. Load Audited Metadata

Load the validated metadata containing the fixed patient-grouped cross-validation fold assignments.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/PAD_UFES_Capstone"
)

DATA_PATH = (
    PROJECT_ROOT
    / "metadata"
    / "pad_ufes_20_with_folds.csv"
)

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (2298, 28)


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,cancer_history,has_piped_water,has_sewage_system,fitspatrick,region,diameter_1,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed,missing_field_count,fold
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ARM,NaN,NaN,NEV,False,False,False,False,False,False,PAT_1516_1765_530.png,False,13,0
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,True,True,True,3.0,NECK,6.0,5.0,BCC,True,True,False,True,True,True,PAT_46_881_939.png,True,0,4
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FACE,NaN,NaN,ACK,True,False,False,False,False,False,PAT_1545_1867_547.png,False,13,0
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HAND,NaN,NaN,ACK,True,False,False,False,False,False,PAT_1989_4061_934.png,False,13,0
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,False,False,False,1.0,FOREARM,5.0,5.0,BCC,True,True,False,False,True,True,PAT_684_1302_588.png,True,0,0


## 3. Inspect Selected Metadata Features

Inspect the data types, unique values, and basic distributions of the selected clinical variables before defining the preprocessing strategy.

In [4]:
SAFE_METADATA = [
    "age",
    "region",
    "itch",
    "grew",
    "hurt",
    "changed",
    "bleed",
    "elevation",
]

feature_summary = pd.DataFrame({
    "dtype": df[SAFE_METADATA].dtypes,
    "unique_values": df[SAFE_METADATA].nunique(dropna=False),
    "missing_values": df[SAFE_METADATA].isna().sum(),
})

display(feature_summary)

,dtype,unique_values,missing_values
age,int64,84,0
region,object,14,0
itch,object,3,0
grew,object,3,0
hurt,object,3,0
changed,object,3,0
bleed,object,3,0
elevation,object,3,0


### Inspect Feature Values

Display the observed values for each selected feature to determine which variables should be treated as numerical or categorical.

In [5]:
for feature in SAFE_METADATA:
    print(f"\n### {feature}")

    display(
        df[feature]
        .value_counts(dropna=False)
        .to_frame("count")
    )


### age


,count
age,
55,73
73,73
58,70
71,66
62,66
...,...
8,2
24,2
12,2



### region


,count
region,
FACE,570
FOREARM,392
CHEST,280
BACK,248
ARM,192
NOSE,158
HAND,126
NECK,93
EAR,73



### itch


,count
itch,
True,1455
False,837
UNK,6


... (6 outputs left out)


### bleed


,count
bleed,
False,1678
True,614
UNK,6



### elevation


,count
elevation,
True,1433
False,863
UNK,2


## 4. Define Metadata Feature Types

The selected metadata contains one continuous numerical feature (`age`), one multi-category anatomical feature (`region`), and six clinical symptom variables.

The symptom variables contain three observed states: `True`, `False`, and `UNK`. `UNK` is retained as a separate category rather than being interpreted as `False`, preserving the distinction between an unknown response and the absence of a symptom.

The preprocessing strategy is therefore:

- standardize `age`,
- one-hot encode `region`,
- one-hot encode the six clinical symptom variables, including `UNK` as a valid category.

All preprocessing will later be fitted using training data only within each cross-validation fold.

In [6]:
NUMERIC_FEATURES = [
    "age",
]

CATEGORICAL_FEATURES = [
    "region",
    "itch",
    "grew",
    "hurt",
    "changed",
    "bleed",
    "elevation",
]

print("Numeric features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

Numeric features:
['age']

Categorical features:
['region', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']


## 5. Validate Categorical Values

Inspect the complete set of anatomical-region categories and verify that categorical values are consistently represented before encoding.

In [7]:
for feature in CATEGORICAL_FEATURES:
    values = sorted(
        df[feature]
        .astype(str)
        .unique()
    )

    print(f"{feature}: {values}")

region: ['ABDOMEN', 'ARM', 'BACK', 'CHEST', 'EAR', 'FACE', 'FOOT', 'FOREARM', 'HAND', 'LIP', 'NECK', 'NOSE', 'SCALP', 'THIGH']
itch: ['False', 'True', 'UNK']
grew: ['False', 'True', 'UNK']
hurt: ['False', 'True', 'UNK']
changed: ['False', 'True', 'UNK']
bleed: ['False', 'True', 'UNK']
elevation: ['False', 'True', 'UNK']


## 6. Define Fold-Safe Metadata Preprocessing

A preprocessing pipeline is defined for the selected clinical features:

- `age` is standardized using the mean and standard deviation of the training partition.
- `region` and the six symptom variables are one-hot encoded.
- `UNK` is retained as a valid categorical state.
- Unknown categories encountered during validation are safely ignored.

The preprocessor must be fitted separately on the training partition of each cross-validation fold to prevent validation information from influencing feature transformation.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_metadata_preprocessor():
    """Create a fresh metadata preprocessor for one training fold."""

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                NUMERIC_FEATURES,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

## 7. Test Preprocessing on One Cross-Validation Fold

Use Fold 0 as a validation fold to verify the preprocessing pipeline.

The preprocessor is fitted only on the remaining training folds and then applied separately to the training and validation partitions.

In [9]:
VALIDATION_FOLD = 0

train_df = df[df["fold"] != VALIDATION_FOLD].copy()
val_df = df[df["fold"] == VALIDATION_FOLD].copy()

preprocessor = build_metadata_preprocessor()

X_train = preprocessor.fit_transform(
    train_df[SAFE_METADATA]
)

X_val = preprocessor.transform(
    val_df[SAFE_METADATA]
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

print("\nTransformed training shape:", X_train.shape)
print("Transformed validation shape:", X_val.shape)

Training samples: 1838
Validation samples: 460

Transformed training shape: (1838, 33)
Transformed validation shape: (460, 33)


## 8. Inspect Transformed Metadata Features

Inspect the names of the transformed metadata features and verify that the preprocessing pipeline produces valid numerical values.

This confirms that the feature representation is suitable for use as input to the multimodal models.

In [10]:
feature_names = preprocessor.get_feature_names_out()

print(f"Number of transformed features: {len(feature_names)}")

for i, feature in enumerate(feature_names, start=1):
    print(f"{i:02d}. {feature}")

Number of transformed features: 33
01. age
02. region_ABDOMEN
03. region_ARM
04. region_BACK
05. region_CHEST
06. region_EAR
07. region_FACE
08. region_FOOT
09. region_FOREARM
10. region_HAND
11. region_LIP
12. region_NECK
13. region_NOSE
14. region_SCALP
15. region_THIGH
16. itch_False
17. itch_True
18. itch_UNK
19. grew_False
20. grew_True
21. grew_UNK
22. hurt_False
23. hurt_True
24. hurt_UNK
25. changed_False
26. changed_True
27. changed_UNK
28. bleed_False
29. bleed_True
30. bleed_UNK
31. elevation_False
32. elevation_True
33. elevation_UNK


## 9. Validate Transformed Values

Check that the transformed training and validation metadata contain only finite numerical values.

In [11]:
print("Training NaN values:", np.isnan(X_train).sum())
print("Validation NaN values:", np.isnan(X_val).sum())

print("Training infinite values:", np.isinf(X_train).sum())
print("Validation infinite values:", np.isinf(X_val).sum())

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

print("\n✓ Metadata preprocessing validated successfully.")

Training NaN values: 0
Validation NaN values: 0
Training infinite values: 0
Validation infinite values: 0

✓ Metadata preprocessing validated successfully.


## 10. Validate Preprocessing Across All Cross-Validation Folds

Apply the metadata preprocessing procedure independently to each fold.

For every fold, the preprocessor is fitted only on the corresponding training partition and then applied to the validation partition. This confirms that all folds produce valid and consistent numerical representations without data leakage.

In [12]:
fold_results = []

for fold in sorted(df["fold"].unique()):
    train_df = df[df["fold"] != fold].copy()
    val_df = df[df["fold"] == fold].copy()

    preprocessor = build_metadata_preprocessor()

    X_train = preprocessor.fit_transform(
        train_df[SAFE_METADATA]
    )

    X_val = preprocessor.transform(
        val_df[SAFE_METADATA]
    )

    fold_results.append({
        "fold": fold,
        "train_samples": len(train_df),
        "validation_samples": len(val_df),
        "train_features": X_train.shape[1],
        "validation_features": X_val.shape[1],
        "train_finite": np.isfinite(X_train).all(),
        "validation_finite": np.isfinite(X_val).all(),
    })

fold_results = pd.DataFrame(fold_results)

display(fold_results)

,fold,train_samples,validation_samples,train_features,validation_features,train_finite,validation_finite
0,0,1838,460,33,33,True,True
1,1,1857,441,33,33,True,True
2,2,1850,448,33,33,True,True
3,3,1839,459,33,33,True,True
4,4,1808,490,33,33,True,True


### Validate Consistency Across Folds

Confirm that every fold produces the same number of transformed metadata features and contains only finite numerical values.

In [13]:
assert fold_results["train_features"].nunique() == 1
assert fold_results["validation_features"].nunique() == 1

assert fold_results["train_finite"].all()
assert fold_results["validation_finite"].all()

print(
    "Metadata dimensions:",
    fold_results["train_features"].iloc[0]
)

print("\n✓ Preprocessing is consistent across all five folds.")

Metadata dimensions: 33

✓ Preprocessing is consistent across all five folds.


## 11. Define Diagnostic Class Mapping

Define a fixed numerical mapping for the six diagnostic classes.

The same class ordering will be used across all benchmark and proposed models to ensure consistent training, probability outputs, confusion matrices, and per-class evaluation.

In [14]:
CLASS_NAMES = sorted(df["diagnostic"].unique())

CLASS_TO_IDX = {
    label: idx
    for idx, label in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    idx: label
    for label, idx in CLASS_TO_IDX.items()
}

print("Class mapping:")
for label, idx in CLASS_TO_IDX.items():
    print(f"{idx}: {label}")

Class mapping:
0: ACK
1: BCC
2: MEL
3: NEV
4: SCC
5: SEK


## 12. Validate Target Encoding

Encode the diagnostic labels using the fixed class mapping and confirm that all samples receive a valid target index.

In [15]:
df["target"] = df["diagnostic"].map(CLASS_TO_IDX)

print(df[["diagnostic", "target"]].drop_duplicates().sort_values("target"))

assert df["target"].notna().all()
assert df["target"].between(0, len(CLASS_NAMES) - 1).all()

print("\n✓ Target encoding validated successfully.")

   diagnostic  target
2         ACK       0
1         BCC       1
53        MEL       2
0         NEV       3
16        SCC       4
5         SEK       5

✓ Target encoding validated successfully.


## 13. Reusable Metadata Preprocessing Configuration

The metadata preprocessing configuration is retained in reusable form rather than saving globally transformed features.

During model training, each cross-validation fold will:

1. create a fresh metadata preprocessor,
2. fit it using the training partition only,
3. transform the training and validation metadata separately,
4. use the same fixed class mapping for target encoding.

This avoids preprocessing leakage and ensures that all multimodal models use the same clinical feature representation.

In [16]:
METADATA_CONFIG = {
    "safe_metadata": SAFE_METADATA,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "num_transformed_features": 33,
    "class_names": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
}

METADATA_CONFIG

{'safe_metadata': ['age',
  'region',
  'itch',
  'grew',
  'hurt',
  'changed',
  'bleed',
  'elevation'],
 'numeric_features': ['age'],
 'categorical_features': ['region',
  'itch',
  'grew',
  'hurt',
  'changed',
  'bleed',
  'elevation'],
 'num_transformed_features': 33,
 'class_names': ['ACK', 'BCC', 'MEL', 'NEV', 'SCC', 'SEK'],
 'class_to_idx': {'ACK': 0, 'BCC': 1, 'MEL': 2, 'NEV': 3, 'SCC': 4, 'SEK': 5}}

## 14. Save Metadata Feature Specification

Save the validated metadata feature definitions and class mapping for reuse during model development.

In [17]:
import json

METADATA_DIR = PROJECT_ROOT / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SPEC_PATH = METADATA_DIR / "feature_spec.json"

with open(FEATURE_SPEC_PATH, "w") as f:
    json.dump(METADATA_CONFIG, f, indent=4)

print(f"Saved feature specification to:\n{FEATURE_SPEC_PATH}")

Saved feature specification to:
/content/drive/MyDrive/PAD_UFES_Capstone/metadata/feature_spec.json


## 15. Preprocessing Summary

The clinical metadata preprocessing protocol is now fixed.

- Eight clinical metadata variables are used in the primary experiments.
- `age` is treated as a numerical variable and standardized within each training fold.
- `region` is treated as a categorical variable with 14 observed anatomical categories.
- `itch`, `grew`, `hurt`, `changed`, `bleed`, and `elevation` are categorical variables with `True`, `False`, and `UNK` states.
- `UNK` is retained as a distinct category rather than being interpreted as `False`.
- Categorical variables are one-hot encoded.
- The final metadata representation contains **33 features**.
- Preprocessing is fitted independently within each training fold to prevent validation leakage.
- The preprocessing procedure was validated across all five patient-grouped cross-validation folds.
- All transformed values were finite and contained no missing values.
- Diagnostic labels are mapped consistently to six numerical class indices.
- The same preprocessing protocol and class mapping will be reused across all benchmark and proposed multimodal models.